<a href="https://colab.research.google.com/github/BrenoLuna861/Classificacao-Trafego-Redes/blob/main/Aula_04_Alunos_Pr%C3%A1tica_02_Semana_2_AI_Talent_Academy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Baixando as duas bibliotecas que serão utilizadas e que não vem por padrão no Google Colab
!pip install datasets tiktoken -q

In [ ]:
def demonstrar_tokens(texto, model="gpt-5"):
    encoding = tiktoken.encoding_for_model(model)
    tokens = encoding.encode(texto)
    print(f"Texto: {texto}")
    print(f"Lista de IDs de Tokens: {tokens}")
    print(f"Quantidade de tokens: {len(tokens)}")

    # Decodificando cada token individualmente para mostrar a divisão
    print("Divisão visual:", [encoding.decode([t]) for t in tokens])

# demonstrar_tokens("seu prompt aqui!")

## Sobre o dataset de exemplo (B2W-Reviews01):
O B2W-Reviews01 é um corpus aberto de avaliações de produtos. Ele contém mais de 130 mil avaliações coletadas em 2018 de clientes de comércio eletrônico, coletadas de sites como Americanas e outros e-commerces.

O B2W-Reviews01 oferece informações também sobre o perfil dos avaliadores, como gênero, idade e localização geográfica. O corpus também apresenta dois tipos diferentes de avaliações

In [ ]:
import tiktoken
from datasets import load_dataset

# Baixa e já carrega em memória, cacheado localmente
dataset = load_dataset("ruanchaves/b2w-reviews01", revision="refs/convert/parquet")

# Ver a estrutura
print(dataset)

In [ ]:
# Acessar o split de treino, por exemplo
dados = dataset["train"]
colunas_para_manter = ['submission_date', 'product_id', 'product_name', 'review_title', 'review_text', 'reviewer_birth_year', 'reviewer_gender', 'reviewer_state']

# Converter pra pandas
df = dados.to_pandas()
df = df[colunas_para_manter]
df.head(1)

### Experimento 1: Análise de Sentimento com LLM

Nesta tarefa, vamos pedir para a LLM analisar o texto da avaliação e classificar como **Positivo**, **Negativo** ou **Neutro**. É um ótimo momento para mostrar como o `system_prompt` ajuda a restringir a saída do modelo.

In [ ]:
# Configurando o cliente para realizar as chamadas para a API da OpenAI

from openai import OpenAI
from google.colab import userdata

client = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = userdata.get('NVD_API_KEY') # Adicionar o seu secret com esse nome!
)

In [ ]:
def analisar_sentimento_llm(texto):
    # Prompt de sistema definindo o comportamento da LLM
    system_prompt = """
        **PERSONA**: Você é um analista de sentimentos especializado em e-commerce.
        **CONTEXTO**: Nossa empresa é um e-commerce que recebe muitas revisões de nossos consumidores.
        **OBJETIVOS**: Responda apenas com UMA palavra: Positivo, Negativo ou Neutro. Não responda com nada mais.
    """

    # Chamada da API (usando o client já configurado anteriormente)
    response = client.chat.completions.create(
        model="nvidia/nemotron-3.5-lightning-30b-a3b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Analise o sentimento desta avaliação: {texto}"}
        ],
        temperature=0.1,
        extra_body={"chat_template_kwargs":{"enable_thinking":True}}
    )
    return response.choices[0].message.content.strip()

# Teste com uma amostra
exemplo_review = df.sample(1).iloc[0]['review_text']
print(f"Review: {exemplo_review}")
print(f"Sentimento Previsto: {analisar_sentimento_llm(exemplo_review)}")

### Experimento 2: Agrupamento em Tópicos (Topic Tagging)

Aqui, o objetivo é mostrar como a LLM pode identificar o *assunto principal* (ex: Logística, Qualidade do Produto, Preço) para podermos agrupar comentários similares depois.

In [ ]:
def identificar_topico(texto):
    # Definimos categorias para facilitar o agrupamento posterior
    topicos_sugeridos = "Logística/Entrega, Qualidade do Produto, Preço/Custo-benefício, Atendimento, Usabilidade"

    system_prompt = f"""
        Identifique qual destes tópicos melhor descreve a reclamação: {topicos_sugeridos}. Responda apenas o nome do tópico.
        Só responda com os tópicos listados anteriormente. Sempre responda em português brasileiro. Não invente nenhum tópico diferente.
    """

    response = client.chat.completions.create(
        model="nvidia/nemotron-3.5-lightning-30b-a3b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": texto}
        ],
        temperature=0.1
    )
    return response.choices[0].message.content.strip()

# Demonstração
amostra_topico = df.sample(1)
for i, row in amostra_topico.iterrows():
    topico = identificar_topico(row['review_text'])
    print(f"Texto: {row['review_text'][:110]}...")
    print(f"-> Tópico Identificado: {topico}\n")

# Exercício:
Nesta aula, aprenderemos a utilizar uma LLM via API para automatizar a classificação de problemas reportados por usuários.

### 1. Entendendo a Estrutura da Chamada
Uma chamada de LLM geralmente envolve:
*   **System Prompt**: Define o comportamento do modelo.
*   **User Prompt**: A entrada de dados ou pergunta.
*   **Temperature**: Controla a criatividade vs. precisão.

---

*Os blocos que necessitam de edição estarão com a flag 'ESCREVA AQUI'*

### Importando os dados:
---
Nesta etapa, estamos importando um conjunto de dados de reclamações de e-commerce. O arquivo contém duas abas principais:
1.  **dataset_treino**: Contém o histórico de reclamações com colunas como `title` (título da reclamação), `description` (relato detalhado do cliente).
2.  **problemas**: Uma lista de referência com todas as categorias de problemas mapeadas pela empresa.

Interpretamos esses dados comparando o que o cliente escreveu com a categoria atribuída manualmente, o que nos permite validar se uma Inteligência Artificial consegue realizar essa classificação de forma autônoma e precisa.

In [ ]:
# NÃO EDITE ESSA CÉLULA:
import pandas as pd

# O ID do arquivo extraído do link fornecido
file_id = '1_9MN5HDaESKDSa4CFNpTvqnOwLWPJJds'
direct_link = f'https://drive.google.com/uc?export=download&id={file_id}'

try:
    df = pd.read_excel(direct_link, sheet_name="dataset_treino")
    categorias = pd.read_excel(direct_link, sheet_name="problemas")
    print("Arquivos carregados com sucesso!")
except Exception as e:
    print(f"Erro ao ler o arquivo: {e}")

In [ ]:
# Implemente a lógica de chamada da API

def classificar_reclamacao(titulo, descricao):
    # 1. Coloque nessa lista todas as categorias:
    CATEGORIAS = []            # TODO: implementar [ESCREVA AQUI]

    # 2. Defina aqui o seu System Prompt
    # Dica: Diga ao modelo quem ele é e quais as regras para a resposta.
    SYSTEM_PROMPT = """
        # ESCREVA AQUI - Seu prompt para instruir a LLM com o que ela deve fazer.
    """

    # 3. Monte o prompt do usuário
    USER_PROMPT = ""           # TODO: implementar [ESCREVA AQUI]

    # 4. Modelo NVIDIA utilizado:
    MODELO = "nvidia/nemotron-3.5-lightning-30b-a3b" # [Pode editar caso esteja utilizando outro modelo]

    # 5. Temperatura: quão criativo o modelo precisa ser para essa tarefa?
    TEMPERATURA = 1            # Qual o valor ideal aqui? [ESCREVA AQUI]

    # 6. O modo de reasoning (pensamento estendido) precisa estar ativado?
    THINKING = ""              # True/False [ESCREVA AQUI]

    # 7. Realize a chamada ao client.chat.completions.create()
    #Lembre-se de passar o SYSTEM_PROMPT e o USER_PROMPT
    resposta = client.chat.completions.create(
        model= "TROQUE POR VARIÁVEL",                                # Que variável precisa vir aqui? [ESCREVA AQUI]
        messages=[
            {"role": "system", "content": "TROQUE POR VARIÁVEL"},    # Que variável precisa vir aqui? [ESCREVA AQUI]
            {"role": "user", "content": "TROQUE POR VARIÁVEL"}       # Que variável precisa vir aqui? [ESCREVA AQUI]
        ],
        temperature="TROQUE POR VARIÁVEL",                           # Que variável precisa vir aqui? [ESCREVA AQUI]
        extra_body={"chat_template_kwargs":{"enable_thinking":"TROQUE POR VARIÁVEL"}}, # Que variável precisa vir aqui? [ESCREVA AQUI]
        stream=False
    )

    return "" # TODO: Retorne apenas a categoria prevista [ESCREVA AQUI]

### 2. Teste Prático
Vamos selecionar uma linha aleatória do nosso dataset e ver se o modelo consegue prever a coluna `problem` corretamente.

In [ ]:
# Selecionando um exemplo aleatório para teste
exemplo = df.sample(1).iloc[0]

print(f"--- DADOS DE ENTRADA ---")
print(f"Título: {exemplo['title']}")
print(f"Descrição: {exemplo['description'][:200]}...")

# Chamada da LLM
predicao = classificar_reclamacao(exemplo['title'], exemplo['description'])

print(f"\n--- RESULTADO DA LLM ---")
print(f"Categoria Prevista: {predicao}")

## Perguntas de Reflexão e Validação
---

### 1. Desempenho quantitativo
Execute a célula de teste abaixo para 5 exemplos aleatórios:
- **a)** Como a decisão da IA se comparou ao gabarito original?
- **b)** Em números absolutos, quantos acertos e erros ocorreram?
- **c)** Qual foi a acurácia (porcentagem de acerto) observada?

### 2. Análise qualitativa e confiabilidade
- **a)** Nos casos de erro, a resposta da IA ainda faz sentido? O erro foi por ambiguidade do texto ou falha do modelo?
- **b)** O modelo 'inventou' alguma categoria que não estava na lista original (alucinação)?
- **c)** Você confiaria em automatizar 100% desse processo ou manteria uma revisão humana?

### 3. Impacto de negócio e próximos passos
- **a)** Como essa classificação automática poderia ajudar o time de Customer Experience (CX)?
- **b)** O que acontece se mudarmos a `temperatura` para 0.0? E se aumentarmos para 1.0?
- **c)** Você sente que se houvesse mais alguma feature (coluna do dataset), isso poderia ajudar o modelo a ser mais preciso? Se sim, qual feature você imagina?

In [ ]:
# Dataset com a label verdadeira:
try:
    df_full = pd.read_excel(direct_link, sheet_name="full_dataset")
    categorias = pd.read_excel(direct_link, sheet_name="problemas")
    print("Arquivos carregados com sucesso!")
except Exception as e:
    print(f"Erro ao ler o arquivo: {e}")

In [ ]:
# Código para responder à Pergunta 1 (Batch de 5 testes)
amostra_teste = df_full.sample(5)
acertos = 0

print(f"{'Item':<5} | {'Real':<25} | {'Previsto':<25} | {'Resultado'}")
print("-" * 80)

for i, (idx, row) in enumerate(amostra_teste.iterrows()):
    real = row['problem']
    previsto = classificar_reclamacao(row['title'], row['description'])

    status = "✅" if real.strip().lower() == previsto.strip().lower() else "❌"
    if status == "✅": acertos += 1

    print(f"{i+1:<5} | {real[:25]:<25} | {previsto[:25]:<25} | {status}")

print("-" * 80)
print(f"Total de Acertos: {acertos} de 5")
print(f"Acurácia: {(acertos/5)*100}%")